# Learning embeddings: flat and structural formulation

A small social network: 14 people, some facts about who is friends with whom, who smokes and who has cancer, and general rules such as "friends of smokers smoke". Many facts are unknown. Each person is a **learned embedding**, and the predicates are grounded as neural networks on the embeddings; training makes facts and rules true together, so the rules fill in the unknown facts. Afterwards, the model is queried with formulas that are not in the knowledge base.

The problem is written twice: in **Part A** in flat form (one constant per person, one formula per fact), in **Part B** with a structural dimension over the people (Tutorial 5), where each group of facts is a single guarded formula. Both parts start from the same parameters, so the results can be compared directly.

The difference is **vectorization**. The flat form is only partly vectorized: rules over variables are batched (`∀ x, y` is one network call on all 196 pairs), but facts are enumerated, one formula and one call per atom. The structural form is vectorized throughout: each fact group is one call on the whole domain, with a mask selecting the facts.

In [ ]:
import copy
import math
import matplotlib.pyplot as plt
import numpy as np
import torch
import sltn
from sltn import Signature, Interpretation, KB
from sltn.interpretation.apply import apply_sat_agg
from sltn.interpretation.operators import AggregPMean, AggregPMeanError

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Data

Two groups, A–H and I–N. Friendships are known inside each group (pairs not listed are not friends) and unknown across groups. Smoking is known for everyone, cancer only for the first group.

In [ ]:
people = list("ABCDEFGHIJKLMN")
group1, group2 = set("ABCDEFGH"), set("IJKLMN")
friends = [("A", "B"), ("A", "E"), ("A", "F"), ("A", "G"), ("B", "C"), ("C", "D"),
           ("E", "F"), ("G", "H"), ("I", "J"), ("J", "M"), ("K", "L"), ("M", "N")]
smokes = {"A", "E", "F", "G", "J", "N"}
cancer = {"A", "E"}

# Part A — Flat formulation

## 1 — Signature

- a sort `Person`, with a constant for each person;
- predicates `Smokes(x)`, `Cancer(x)`, `Friends(x, y)`;
- variables `x`, `y` over people.

In [ ]:
sig = Signature("smokers")
sig.sort("Person")
for person in people:
    sig.constant(person, "Person")
sig.predicate("Smokes", ["Person"])
sig.predicate("Cancer", ["Person"])
sig.predicate("Friends", ["Person", "Person"])
sig.variable("x", "Person")
sig.variable("y", "Person")
print(sig)

## 2 — Knowledge base

**Facts** are ground atoms, true or negated: `Friends(A, B)`, `¬Friends(A, C)`, `Smokes(A)`, `¬Cancer(B)`, … . The facts of each kind (positive friendships, negative friendships, …) form their own KB, aggregated on its own; SatAgg then combines the 6 groups and the 6 rules. In this way the 31 negative friendships do not outweigh the 12 positive ones by their number alone. `KB` puts the clauses of a nested KB directly in the outer list (`KB([KB([f1, f2]), r])` is `KB([f1, f2, r])`), so the outer SatAgg is applied explicitly (§5).

**Rules**:

```
∀ x: ¬Friends(x, x)                                   nobody is their own friend
∀ x, y: (Friends(x, y) -> Friends(y, x))              friendship is symmetric
∀ x: ∃ y: Friends(x, y)                               everyone has a friend
∀ x, y: ((Friends(x, y) & Smokes(x)) -> Smokes(y))    friends of smokers smoke
∀ x: (Smokes(x) -> Cancer(x))                         smoking causes cancer
∀ x: (¬Smokes(x) -> ¬Cancer(x))                       not smoking, no cancer
```

The rules are soft: "friends of smokers smoke" is contradicted by the facts (B is a friend of A, who smokes, but B does not), and training finds a trade-off.

**Queries**, not in the knowledge base:
- φ1 `∀ x: (Cancer(x) -> Smokes(x))`: "people with cancer smoke";
- φ2 `∀ x, y: ((Cancer(x) & Cancer(y)) -> Friends(x, y))`: "people with cancer are friends with each other".

In [ ]:
facts = {
    "Friends+": [f"Friends({u}, {v})" for u, v in friends],
    "Friends-": [f"¬Friends({u}, {v})" for group in (group1, group2) for u in sorted(group) for v in sorted(group)
                 if u < v and (u, v) not in friends],
    "Smokes+": [f"Smokes({p})" for p in people if p in smokes],
    "Smokes-": [f"¬Smokes({p})" for p in people if p not in smokes],
    "Cancer+": [f"Cancer({p})" for p in people if p in cancer],
    "Cancer-": [f"¬Cancer({p})" for p in sorted(group1) if p not in cancer],
}
rules = [
    "∀ x: ¬Friends(x, x)",
    "∀ x, y: (Friends(x, y) -> Friends(y, x))",
    "∀ x: ∃ y: Friends(x, y)",
    "∀ x, y: ((Friends(x, y) & Smokes(x)) -> Smokes(y))",
    "∀ x: (Smokes(x) -> Cancer(x))",
    "∀ x: (¬Smokes(x) -> ¬Cancer(x))",
]
clauses = [KB([sig.parse(f) for f in group]) for group in facts.values()] + [sig.parse(r) for r in rules]
queries = {
    "φ1 Cancer -> Smokes": sig.parse("∀ x: (Cancer(x) -> Smokes(x))"),
    "φ2 Cancer & Cancer -> Friends": sig.parse("∀ x, y: ((Cancer(x) & Cancer(y)) -> Friends(x, y))"),
}
print({name: len(group) for name, group in facts.items()})

## 3 — Model

Each person is an embedding, a learned `nn.Parameter` of size 5 grounding its constant. `Smokes` and `Cancer` are small MLPs with a sigmoid on the embedding, `Friends` the same on the two concatenated embeddings.

In [ ]:
embedding_size = 5

class Predicate(torch.nn.Module):
    def __init__(self, arity):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(arity * embedding_size, 16), torch.nn.ELU(),
            torch.nn.Linear(16, 1), torch.nn.Sigmoid(),
        )

    def forward(self, *args):
        return self.net(torch.cat(torch.broadcast_tensors(*args), dim=-1))

## 4 — Interpretation

The constants are the embeddings, and the variables `x`, `y` are all of them stacked, rebuilt at each step from the current values. Since `x` and `y` are the same 14 people, `∀ x, y` ranges over all 196 pairs.

The operators are the defaults; the implication is Reichenbach, $1 - a + ab$, which gives gradient to both sides (Tutorial 3). The initial parameters are saved, to start Part B from the same point.

In [ ]:
interp = Interpretation(sig).to(device)
interp["Person"] = sltn.Type("Person", shape=(embedding_size,))
for person in people:
    interp[person] = torch.nn.Parameter(torch.rand(embedding_size, device=device))
interp["Smokes"] = Predicate(1)
interp["Cancer"] = Predicate(1)
interp["Friends"] = Predicate(2)


def ground_people():
    everyone = torch.stack([interp[p] for p in people])
    interp["x"] = everyone
    interp["y"] = everyone
    return everyone


initial_embeddings = torch.stack([interp[p].detach().clone() for p in people])
initial_predicates = {name: copy.deepcopy(interp[name].state_dict()) for name in ("Smokes", "Cancer", "Friends")}

## 5 — Training

The loss is $1 - \text{truth}(\mathcal{K})$, on all the people at once: `satisfaction()` evaluates each fact group and rule and combines them with SatAgg. The aggregators start lenient (p = 1, the mean) and become stricter (p = 6) after 200 epochs, so that "everyone has a friend" requires one friend rather than a partial friendship with everyone.

In [ ]:
def satisfaction():
    return apply_sat_agg(interp.logic["SatAgg"], *(interp(clause) for clause in clauses)).value


initial_results = {}
with torch.no_grad():
    ground_people()
    for p in (1, 6):
        interp.logic["∀"].p = interp.logic["∃"].p = interp.logic["SatAgg"].p = p
        initial_results[p] = (torch.stack([interp(clause).value.squeeze() for clause in clauses]), satisfaction().item())

optimizer = torch.optim.Adam(interp.parameters(), lr=0.01)

for epoch in range(2000):
    p = 1 if epoch <= 200 else 6
    interp.logic["∀"].p = interp.logic["∃"].p = interp.logic["SatAgg"].p = p
    ground_people()
    optimizer.zero_grad()
    loss = 1 - satisfaction()
    loss.backward()
    optimizer.step()
    if epoch % 400 == 0 or epoch == 1999:
        with torch.no_grad():
            ground_people()
            q = " | ".join(f"{name.split()[0]} {interp(phi).value.item():.3f}" for name, phi in queries.items())
            print(f"epoch {epoch:4d} | sat {satisfaction().item():.3f} | {q}")

## 6 — Results

Known facts (white: unknown) next to what the model learned.

In [ ]:
with torch.no_grad():
    everyone = ground_people()
    smokes_pred = interp["Smokes"](everyone).squeeze(-1).cpu()
    cancer_pred = interp["Cancer"](everyone).squeeze(-1).cpu()
    n = len(people)
    friends_pred = interp["Friends"](everyone.repeat_interleave(n, 0), everyone.repeat(n, 1)).reshape(n, n).cpu()

nan = math.nan
group = lambda p: 1 if p in group1 else 2
known_sc = np.array([[p in smokes, (p in cancer) if p in group1 else nan] for p in people], dtype=float)
known_friends = np.array([[nan if u == v or group(u) != group(v) else float((u, v) in friends or (v, u) in friends)
                           for v in people] for u in people])
learned_friends = friends_pred.numpy().copy()
np.fill_diagonal(learned_friends, nan)

fig, axes = plt.subplots(1, 4, figsize=(16, 4), gridspec_kw={"width_ratios": [1, 3, 1, 3]})
panels = [(known_sc, "Known: Smokes, Cancer"), (known_friends, "Known: Friends"),
          (torch.stack([smokes_pred, cancer_pred], 1).numpy(), "Learned: Smokes, Cancer"), (learned_friends, "Learned: Friends")]
for ax, (values, title) in zip(axes, panels):
    im = ax.imshow(values, vmin=0, vmax=1, aspect="auto")
    ax.set_title(title)
    ax.set_yticks(range(n), people)
    ax.set_xticks(range(values.shape[1]), ["S", "C"] if values.shape[1] == 2 else people)
fig.colorbar(im, ax=axes)
plt.show()

print("Cancer, group 2 (unknown):", {p: round(cancer_pred[i].item(), 2) for i, p in enumerate(people) if p in group2})

The truth value of each fact group and rule after training:

In [ ]:
with torch.no_grad():
    ground_people()
    for name, clause in zip(list(facts) + rules, clauses):
        print(f"{interp(clause).value.item():.3f}  {name}")

The learned heatmaps match the known facts. The unknown facts are filled in by the rules: in the second group cancer is unknown for everyone, and the model gives it to the two smokers J and N and to nobody else, by "smoking causes cancer" and its converse.

The knowledge base is not fully satisfied: some rules contradict the facts, and with p = 6 the aggregators are close to the minimum, so the worst clauses weigh most.

φ1 "people with cancer smoke" comes out true, as the rules suggest. φ2 "people with cancer are friends with each other" is implied by nothing and stays low.

# Part B — The same problem with a structural dimension

The people become the positions of a dimension `P`. All embeddings are one constant `x: Person[P]`, a 14 × 5 matrix, and the facts are 0/1 masks over `P` (structural relations). Each fact group becomes one guarded formula:

| Part A (flat) | Part B (structural) |
|---|---|
| a constant per person `A`, `B`, … | one constant `x[P]`, positions `p`, `p1` |
| a formula per fact, `Friends(A, B)`, … | one guarded formula `∀ p, p1 \| friends_pos(p, p1): Friends(x[p], x[p1])` |
| `∀ x, y` over the stacked constants | `∀ p, p1` over the positions |
| `∀ x: ¬Friends(x, x)` | `∀ p, p1 \| same(p, p1): ¬Friends(x[p], x[p1])` (identity mask) |

A guarded ∀ with a p-mean aggregator is the same mean as the fact group of Part A, so the two KBs have the same 12 clauses with the same values.

In [ ]:
n = len(people)
idx = {person: i for i, person in enumerate(people)}
friends_pos, friends_neg = torch.zeros(n, n), torch.zeros(n, n)
for u, v in friends:
    friends_pos[idx[u], idx[v]] = 1
for group in (group1, group2):
    for u in group:
        for v in group:
            if u < v and (u, v) not in friends:
                friends_neg[idx[u], idx[v]] = 1
smoker = torch.tensor([float(person in smokes) for person in people])
cancer_pos = torch.tensor([float(person in cancer) for person in people])
cancer_neg = torch.tensor([float(person in group1 and person not in cancer) for person in people])

sig_struct = Signature("smokers_struct")
sig_struct.sort("Person")
sig_struct.dimension("P")
sig_struct.structural_variable("p", "P")
sig_struct.constant("x", "Person", dims=["P"])
sig_struct.predicate("Smokes", ["Person"])
sig_struct.predicate("Cancer", ["Person"])
sig_struct.predicate("Friends", ["Person", "Person"])
for name in ("friends_pos", "friends_neg", "same"):
    sig_struct.structural_relation(name, ["P", "P"])
for name in ("smoker", "nonsmoker", "cancer_pos", "cancer_neg"):
    sig_struct.structural_relation(name, ["P"])
print(sig_struct)

## Structural knowledge base

`Smokes(x[p])` applies `Smokes` at every position: it is "p smokes", with a free axis `p`. `Friends(x[p], x[p1])` is the 14 × 14 matrix of all pairs.

In [ ]:
F = "Friends(x[p], x[p1])"
kb_struct = KB([sig_struct.parse(f) for f in [
    f"∀ p, p1 | friends_pos(p, p1): {F}",
    f"∀ p, p1 | friends_neg(p, p1): ¬{F}",
    "∀ p | smoker(p): Smokes(x[p])",
    "∀ p | nonsmoker(p): ¬Smokes(x[p])",
    "∀ p | cancer_pos(p): Cancer(x[p])",
    "∀ p | cancer_neg(p): ¬Cancer(x[p])",
    f"∀ p, p1 | same(p, p1): ¬{F}",
    f"∀ p, p1: ({F} -> Friends(x[p1], x[p]))",
    f"∀ p: ∃ p1: {F}",
    f"∀ p, p1: (({F} & Smokes(x[p])) -> Smokes(x[p1]))",
    "∀ p: (Smokes(x[p]) -> Cancer(x[p]))",
    "∀ p: (¬Smokes(x[p]) -> ¬Cancer(x[p]))",
]])
queries_struct = {
    "φ1 Cancer -> Smokes": sig_struct.parse("∀ p: (Cancer(x[p]) -> Smokes(x[p]))"),
    "φ2 Cancer & Cancer -> Friends": sig_struct.parse(f"∀ p, p1: ((Cancer(x[p]) & Cancer(x[p1])) -> {F})"),
}

## Interpretation

Structural quantifiers over `P` default to copies of ∀ and ∃; here we set `logic["∀,P"]` and `logic["∃,P"]` explicitly to p = 1, as ∀ and ∃ in Part A. The embeddings and the predicate networks are copies of the **initial** parameters of Part A. Before training, the 12 clauses and the KB have exactly the same values in both formulations, for both p = 1 and p = 6.

In [ ]:
interp_struct = Interpretation(sig_struct).to(device)
interp_struct.logic["∀,P"] = AggregPMeanError(p=1)
interp_struct.logic["∃,P"] = AggregPMean(p=1)
interp_struct.logic["SatAgg"].p = 1
interp_struct["Person"] = sltn.Type("Person", shape=(embedding_size,))
interp_struct["x"] = torch.nn.Parameter(initial_embeddings.clone())
for name, arity in (("Smokes", 1), ("Cancer", 1), ("Friends", 2)):
    model = Predicate(arity).to(device)
    model.load_state_dict(initial_predicates[name])
    interp_struct[name] = model
for name, mask in (("friends_pos", friends_pos), ("friends_neg", friends_neg), ("same", torch.eye(n)),
                   ("smoker", smoker), ("nonsmoker", 1 - smoker), ("cancer_pos", cancer_pos), ("cancer_neg", cancer_neg)):
    interp_struct[name] = mask.to(device)

with torch.no_grad():
    for p in (1, 6):
        interp_struct.logic["∀,P"].p = interp_struct.logic["∃,P"].p = interp_struct.logic["SatAgg"].p = p
        struct_clauses = torch.stack([interp_struct(clause).value.squeeze() for clause in kb_struct.clauses])
        struct_initial_sat = interp_struct(kb_struct).value.item()
        initial_clauses, initial_sat = initial_results[p]
        torch.testing.assert_close(initial_clauses, struct_clauses, rtol=1e-5, atol=1e-6)
        torch.testing.assert_close(torch.tensor(initial_sat), torch.tensor(struct_initial_sat), rtol=1e-5, atol=1e-6)
        print(f"p={p}: initial satisfaction flat {initial_sat:.6f}, structural {struct_initial_sat:.6f}; "
              f"maximum clause difference {(initial_clauses - struct_clauses).abs().max().item():.2g}")

## Training and comparison

Same optimizer and schedule as in Part A. There are no variables to ground: `x` is the parameter itself.

In [ ]:
optimizer_struct = torch.optim.Adam(interp_struct.parameters(), lr=0.01)
aggregators_struct = [interp_struct.logic["∀,P"], interp_struct.logic["∃,P"], interp_struct.logic["SatAgg"]]
for epoch in range(2000):
    for agg in aggregators_struct:
        agg.p = 1 if epoch <= 200 else 6
    optimizer_struct.zero_grad()
    loss = 1 - interp_struct(kb_struct).value
    loss.backward()
    optimizer_struct.step()
    if epoch % 400 == 0 or epoch == 1999:
        with torch.no_grad():
            q = " | ".join(f"{name.split()[0]} {interp_struct(phi).value.item():.3f}" for name, phi in queries_struct.items())
            print(f"epoch {epoch:4d} | sat {interp_struct(kb_struct).value.item():.3f} | {q}")

In [ ]:
with torch.no_grad():
    ground_people()
    smokes_struct = interp_struct("Smokes(x[p])").value.flatten().cpu()
    cancer_struct = interp_struct("Cancer(x[p])").value.flatten().cpu()
    friends_struct = interp_struct(F).value.reshape(n, n).cpu()
    print(f"Final KB satisfaction: flat {satisfaction().item():.4f}, structural {interp_struct(kb_struct).value.item():.4f}")
    for name in queries:
        print(f"{name}: flat {interp(queries[name]).value.item():.4f}, structural {interp_struct(queries_struct[name]).value.item():.4f}")
    for name, flat, structural in (("Smokes", smokes_pred, smokes_struct), ("Cancer", cancer_pred, cancer_struct), ("Friends", friends_pred, friends_struct)):
        print(f"{name}: maximum prediction difference {(flat - structural).abs().max().item():.4f}")
    print("Cancer, group 2 (unknown):", {p: round(cancer_struct[i].item(), 2) for i, p in enumerate(people) if p in group2})
    print("\nClause satisfaction: flat | structural")
    for name, flat, structural in zip(list(facts) + rules, clauses, kb_struct.clauses):
        print(f"{interp(flat).value.item():.3f} | {interp_struct(structural).value.item():.3f}  {name}")

The two models stay almost identical through training.

## Summary

- Individuals are grounded by learned embeddings and predicates are networks on them; facts and rules train both, and the rules fill in the unknown facts.
- Each fact group is aggregated on its own and counts as one term in SatAgg, like a rule, whatever its number of facts (a nested `KB` alone would not do this: its clauses simply join the outer list).
- With a structural dimension, the embeddings are one tensor, facts are masks and each fact group is one guarded formula. The semantics is the same, but the facts are vectorized too.